## Usage example

Here is an example of a LSA pipeline that:
1. Ingests a collection of texts
2. Makes the corresponding document-term matrix using stemming and removing stop words
3. Extracts 40 topics
4. Shows a table with the extracted topics
5. Shows a table with statistical thesaurus entries for selected words  

In [ ]:
import random
from LatentSemanticAnalyzer.LatentSemanticAnalyzer import *
from LatentSemanticAnalyzer.DataLoaders import *
from OutlierIdentifiers import *
import snowballstemmer

In [ ]:
# Collection of texts
dfAbstracts = load_abstracts_data_frame()
docs = dict(zip(dfAbstracts.ID, dfAbstracts.Abstract))
len(docs)

In [ ]:
# Stemmer object (to preprocess words in the pipeline below)
stemmerObj = snowballstemmer.stemmer("english")

In [ ]:
# Words to show statistical thesaurus entries for
words = ["notebook", "computational", "function", "neural", "talk", "programming"]

In [ ]:
# Reproducible results
random.seed(12)

In [ ]:
# Remove non-strings
docs2 = { k:v for k, v in docs.items() if isinstance(v, str) }
len(docs2)

In [ ]:
# LSA pipeline
lsaObj = (LatentSemanticAnalyzer()
          .make_document_term_matrix(docs=docs2,
                                     stop_words=True,
                                     stemming_rules=True,
                                     min_length=3)
          .apply_term_weight_functions(global_weight_func="None",
                                       local_weight_func="None",
                                       normalizer_func="Cosine")
          .extract_topics(number_of_topics=40, min_number_of_documents_per_term=10, method="SVD")
          .echo_topics_interpretation(number_of_terms=12, wide_form=True)
          .echo_statistical_thesaurus(terms=stemmerObj.stemWords(words),
                                      wide_form=True,
                                      number_of_nearest_neighbors=12,
                                      method="cosine",
                                      echo_function=lambda x: print(x.to_string())))

In [ ]:
lsaObj.echo_document_term_matrix_statistics();

---

## Find outliers in the topics interpretation data frame

In [ ]:
dfTopicsLongForm = lsaObj.get_topics_interpretation(number_of_terms=120, as_data_frame=True, wide_form=False, echo=False).take_value()
dfTopicsLongForm

In [ ]:
# Group by "Topic" and select rows where Score is an outlier according to your function
dfTopicsOfOutliersLongForm = (
    dfTopicsLongForm.groupby("Topic", group_keys=True)
    .apply(lambda g: g[outlier_identifier(g["Score"].tolist(), identifier = lambda v: top_outliers(hampel_identifier_parameters(v)))])
)

In [ ]:
# Optional: reset index for a clean DataFrame
#dfTopicsOfOutliersLongForm = dfTopicsOfOutliersLongForm.reset_index(drop=True)

In [ ]:
dfTopicsOfOutliersLongForm